[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/10_snow-data/10_02_ua_snow_analysis_mapping.ipynb)

# Module 10, Lesson 2: University of Arizona 800 m Snow Analysis and Mapping
## Equal-area reprojection, resampling choices, subbasin snow water volume, and watershed mapping in the Minnesota River Basin

In this lesson, you load the clipped **University of Arizona (UA) 800 m** daily SWE and snow depth stacks prepared in Lesson 1 (or the published files in `data/`), project them onto an equal-area `EPSG:5070` (`800 m`) grid, compare resampling methods during spring melt, and map 196 days of snow storage across the **Minnesota River Basin (HUC 4 `0702`, 44,050.3 km²)** and its **12 HUC-8 subbasins** (`07020001` to `07020012`) for **WY2023 (`2022-11-01` to `2023-05-15`)**.

### Module 10 at a Glance

| Lesson | Notebook | Focus in Minnesota River HUC 4 (`0702`) |
|---|---|---|
| 1 | `10_01_downloading_snow_data.ipynb` | Live USGS WBD, UA 800 m NetCDF, and SNODAS raw binary `.tar` downloads, format inspection, GeoTIFF stack creation |
| 2 (this notebook) | `10_02_ua_snow_analysis_mapping.ipynb` | UA 800 m `EPSG:5070` reprojection, resampling tests, 196-day HUC-4 and 12-HUC-8 SWE, depth, and volume maps |
| 3 | `10_03_snodas_snow_analysis_mapping.ipynb` | SNODAS 1 km `2013-10-01` grid reconciliation, SWE and `1044` melt runoff analysis, maps, and `SNODAS - UA` comparison |

## Lesson Information

Lesson Level: Intermediate

Purpose: Teach H&H engineers how to project University of Arizona 800 m SWE and snow depth stacks onto an equal-area grid, choose defensible resampling settings, quantify subbasin snow water storage, and export traceable maps and CSV tables.

### Learning Objectives

By the end of this lesson, you will be able to:

1. Explain why native `EPSG:4269` (`0.00833333°`) pixels vary in ground area across latitude and require reprojection to equal-area `EPSG:5070` before computing basin volumes.
2. Implement the buffered-window, reproject, and polygon-mask workflow so watershed edge interpolation keeps full neighbor support.
3. Evaluate `nearest`, `bilinear`, `cubic`, and `average` resampling across active spring meltlines without introducing negative SWE pixels.
4. Compute 196-day SWE, snow depth, bulk snowpack density, snow-covered fraction, and water volume (`km³`, `kAF`) across HUC-4 and HUC-8 subbasins.
5. Export provenance-tagged CSV tables, maps, and GeoTIFF rasters for cross-comparison in Lesson 3.

### Prerequisites

- Lesson 1 (`10_01_downloading_snow_data.ipynb`), or the prepared Module 10 `data/` files
- Familiarity with `rasterio`, `geopandas`, NumPy arrays, and `pandas`

### Inputs and Outputs

| Role | File | Notes |
|---|---|---|
| Input | `minnesota_river_wbd_0702.gpkg` | USGS WBD HUC 4 (`0702`) and 12 HUC-8 subbasins |
| Input | `ua_800m_mn_river_wy2023_swe.tif`, `..._depth.tif` | 196-band native `EPSG:4269` stacks (`int16`, mm, `-999` NoData) |
| Input (optional spot-check) | Live or Lesson 1 cached `2023-04-01` `.nc` | Runs `0 mm` live-vs-stack verification automatically when online |
| Output | `module10_outputs/ua_wy2023_huc04_daily_timeseries.csv` | Daily HUC-4 SWE, depth, bulk density, snow cover, and volume (read by Lesson 3) |
| Output | `module10_outputs/ua_wy2023_huc08_daily_timeseries.csv` | Daily SWE, depth, and snow cover for all 12 HUC-8 subbasins |
| Output | `module10_outputs/ua_wy2023_huc08_peak_summary.csv` | Peak SWE, timing, depth, volume, and meltout date by subbasin |
| Output | `module10_outputs/ua_wy2023_peak_swe_5070_800m.tif` | Clipped peak-date SWE GeoTIFF in `EPSG:5070` (`800 m`) with provenance |
| Output (figures) | `module10_outputs/ua_fig1_*.png` to `ua_fig4_*.png` | Four 300 dpi figures formatted in Arial or Liberation Sans |

## Using AI in This Lesson

An AI coding assistant helps with raster reprojection and subbasin aggregation when you ask it to check units, masks, and physical bounds against printed values.

Keep three habits throughout Lesson 2:

1. Convert `-999` water-mask cells to `np.nan` before casting or reprojecting so `bilinear` interpolation never blends lake flags into surrounding land pixels.
2. Include `0 mm` bare-ground cells inside basin spatial averages so snow-free land stays in the denominator during spring melt.
3. Verify `valid_area_km2` against the `EPSG:5070` WBD polygon area (`44,050.3 km²`) after masking.

Starter questions to try with your AI assistant:

- "Look at my resampling table on `2023-04-12`. Why did `cubic` produce negative SWE values and exceed the raw maximum SWE, while `bilinear` remained strictly inside `[0, raw_max]`?"
- "My `EPSG:4269` pixel area in southern Minnesota is `0.627 km²` and in northern Minnesota is `0.594 km²`, while `800 m x 800 m` equals `0.64 km²`. How much volume bias happens if I skip reprojecting to `EPSG:5070`?" 

## Part 1: Overview of UA 800 m Analysis, Masking, and Projection Rules

### Product Properties That Govern Watershed Calculations

| Property | UA 800 m Specification | Effect on Analysis |
|---|---|---|
| Assimilation inputs | NRCS SNOTEL, NWS COOP depth (via Dawson et al. 2017 snow density), 800 m PRISM weather, and SW-US lidar ANN adjustments | No SNOTEL stations exist in Minnesota, so accuracy relies on COOP depth spacing and PRISM precipitation |
| Native grid and CRS | `EPSG:4269` (NAD83 degrees), `30 arc-sec` (`0.00833333°`) | Pixels measure `~677 m x 926 m` in southern MN and `~641 m x 926 m` in northern MN (~5% smaller than `0.64 km²`) |
| Storage and `_FillValue` | `int16` in `mm` (`scale = 1`), `_FillValue = -999` (lakes, ice, outside CONUS) | Replace `-999` with `np.nan` before calling `reproject()` |
| Observation day | PRISM day ending `12:00 UTC` (`06:00 CST` / `07:00 CDT`) | 6 hours later than SNODAS (`06:00 UTC` snapshot) |
| Release status | `early`, `provisional`, and `stable` | All 196 WY2023 bands in our stack are finalized `stable` files |

### Three-Step Reprojection Sequence

1. Keep the `0.15°` geographic margin prepared in Lesson 1 while calling `rasterio.warp.reproject()` onto a snapped `EPSG:5070` (`800 m`) grid so `2x2` (`bilinear`) or `4x4` (`cubic`) stencils stay complete along the watershed perimeter.
2. Convert `-999` to `np.nan` in `float32` before calling `reproject(..., src_nodata=np.nan, dst_nodata=np.nan)`.
3. Mask out cells outside the `huc4_proj` and `huc8_proj` polygons after reprojection on the `0.64 km²` equal-area grid.

Primary references:

- Broxton, P. D., Moeser, C. D., and Xiong, Z. (2024), Development and evaluation of an 800 m gridded SWE and snow depth dataset over CONUS, *Earth and Space Science*, [doi:10.1029/2023EA002964](https://doi.org/10.1029/2023EA002964)
- Broxton, P., Zeng, X., and Dawson, N. (2019), `NSIDC-0719` User Guide and [DOI: 10.5067/0GGPB220EX6A](https://doi.org/10.5067/0GGPB220EX6A)
- Dawson, N., Broxton, P. D., and Zeng, X. (2017), Snow density parameterization, *J. Hydrometeorology*, [doi:10.1175/JHM-D-16-0166.1](https://doi.org/10.1175/JHM-D-16-0166.1)

## Part 2: Workspace Setup, Colab File Locations, and Loading Data

### Where Files Are Located in Google Colab

When running in Google Colab, your active directory is `/content/`. Click the **Files** folder icon on the left sidebar to browse folders visually:

1. Input files (`minnesota_river_wbd_0702.gpkg`, `ua_800m_mn_river_wy2023_swe.tif`, `ua_800m_mn_river_wy2023_depth.tif`) go inside `/content/data/` (or `/content/drive/MyDrive/module10_snow_data/` when `USE_GOOGLE_DRIVE = True`).
2. Output CSVs, GeoTIFFs, figures, and `.zip` archives are written to `/content/module10_outputs/` (or `/content/drive/MyDrive/module10_snow_data/outputs/` when `USE_GOOGLE_DRIVE = True`).
3. If you generated the full 196-day files in Lesson 1 with `USE_GOOGLE_DRIVE = True`, set `USE_GOOGLE_DRIVE = True` below so Lesson 2 reads them automatically from Google Drive. Otherwise, `ensure_course_files()` opens a single Colab upload prompt where you can select the 3 input files or upload a `.zip` archive.

The first code cell checks and installs any missing Python packages.

In [ ]:
import importlib.util  # Checks if a package can be imported without loading it yet
import subprocess      # Runs 'pip install' from Python when libraries are missing
import sys             # Provides the path to the currently running Python interpreter

# Map {import_name: pip_package_name}
REQUIRED_PACKAGES = {
    'rasterio': 'rasterio',
    'rioxarray': 'rioxarray',
    'geopandas': 'geopandas',
    'xarray': 'xarray',
    'netCDF4': 'netCDF4',
    'cftime': 'cftime',
}

# List comprehension keeps only packages where find_spec(...) returns None (not installed)
missing = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print(f"Installing: {', '.join(missing)}")
    # '*missing' passes each package name as a separate CLI argument, and '-q' keeps pip output short
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print("Done.")
else:
    print("All required packages are already installed.")

### Importing Libraries and Configuring Arial Map Styling

This cell loads array, GIS, raster, and plotting libraries, configures Arial (with Liberation Sans fallback on Linux/Colab) at 300 dpi, and defines reusable scale bar and north arrow map helpers.

In [ ]:
# Standard library
import datetime
import importlib.util
import io
import math
import re
import time
import zipfile
from pathlib import Path

# Numerics and tables
import numpy as np
import pandas as pd
import requests

# Vector and geospatial
import geopandas as gpd
from pyproj import CRS, Geod

# Raster and NetCDF
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import Affine, from_origin
from rasterio.warp import Resampling, reproject
from rasterio.windows import Window, from_bounds as window_from_bounds
import xarray as xr

# Plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

# Enforce Arial (Liberation Sans fallback on Linux/Colab)
FONT_PREFERENCE = ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': FONT_PREFERENCE,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Build set of installed font family names and pick the first available match from FONT_PREFERENCE
installed_fonts = {f.name for f in font_manager.fontManager.ttflist}
active_font = next((f for f in FONT_PREFERENCE if f in installed_fonts), 'DejaVu Sans')
print(f"Active figure font: {active_font}")

# Continuous white-to-blue-to-purple color scale for SWE and snow depth maps
SNOW_CMAP = LinearSegmentedColormap.from_list(
    'ua_snow', ['#F4F8FB', '#D6EAF8', '#85C1E9', '#2E86C1', '#1B4F72', '#4A235A'], N=256
)
SNOW_CMAP.set_bad('white')  # Paint NaN (outside basin or lake mask) solid white

# Diverging red-white-blue color scale centered at zero for resampling differences
DIFF_CMAP = LinearSegmentedColormap.from_list(
    'snow_diff', ['#B03A2E', '#EC7063', '#F8F9F9', '#5DADE2', '#1B4F72'], N=256
)
DIFF_CMAP.set_bad('white')


def add_scalebar(ax, length_km: float = 50.0, location: tuple[float, float] = (0.04, 0.05), show_mid: bool = True):
    """Add an alternating black/white metric scale bar to an EPSG:5070 axis (in meters)."""
    # Read map limits (meters in EPSG:5070) and place bar using fractional anchor offsets
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)
    bar_len = length_km * 1000.0        # Convert km length to meters
    bar_h = 0.015 * (ymax - ymin)       # Bar thickness set to 1.5% of axis height
    half = bar_len / 2.0
    ax.add_patch(Rectangle((x0, y0), half, bar_h, facecolor='#1C2833', edgecolor='#1C2833', lw=0.8, zorder=10))
    ax.add_patch(Rectangle((x0 + half, y0), half, bar_h, facecolor='white', edgecolor='#1C2833', lw=0.8, zorder=10))
    ax.text(x0, y0 + bar_h * 1.4, '0', ha='center', va='bottom', fontsize=8, fontweight='bold', zorder=10)
    if show_mid:
        ax.text(x0 + half, y0 + bar_h * 1.4, f'{int(length_km / 2)}', ha='center', va='bottom', fontsize=8, zorder=10)
    ax.text(x0 + bar_len, y0 + bar_h * 1.4, f'{int(length_km)} km', ha='center', va='bottom', fontsize=8, fontweight='bold', zorder=10)


def add_north_arrow(ax, x_frac: float = 0.94, y_frac: float = 0.88):
    """Add a cartographic north arrow in axes fraction coordinates."""
    # Place arrow using relative (0 to 1) axis coordinates so layout stays fixed across zoom levels
    ax.annotate(
        'N', xy=(x_frac, y_frac + 0.07), xytext=(x_frac, y_frac),
        xycoords='axes fraction', textcoords='axes fraction',
        ha='center', va='center', fontsize=10, fontweight='bold', color='#1C2833',
        arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#1C2833', mutation_scale=13), zorder=10,
    )


def format_km_axes(ax):
    """Format EPSG:5070 meter axes as clean kilometers."""
    # FuncFormatter divides meter tick numbers by 1,000 for readable kilometer ticks
    fmt = mpl.ticker.FuncFormatter(lambda v, pos: f'{v / 1000:.0f}')
    ax.xaxis.set_major_formatter(fmt)
    ax.yaxis.set_major_formatter(fmt)
    ax.set_xlabel('Albers Easting (km, EPSG:5070)')
    ax.set_ylabel('Albers Northing (km, EPSG:5070)')


print(f"Versions: numpy {np.__version__}, pandas {pd.__version__}, "
      f"geopandas {gpd.__version__}, rasterio {rasterio.__version__}")

### Verifying or Uploading Lesson 2 Input Files

The cell below configures `DATA_DIR` and `OUTPUT_DIR` and verifies that `minnesota_river_wbd_0702.gpkg`, `ua_800m_mn_river_wy2023_swe.tif`, and `ua_800m_mn_river_wy2023_depth.tif` exist on disk.

In [ ]:
# Analysis grid and UA dataset constants
TARGET_HUC4 = '0702'
TARGET_CRS = 'EPSG:5070'       # Equal-area projection in meters (each 800 m cell = 0.64 km^2)
TARGET_RES_M = 800.0
BBOX_PAD_DEG = 0.15            # 0.15-degree margin kept around the basin from Lesson 1
UA_NODATA = -999
UA_BASE_URL = 'https://climate.arizona.edu/data/UA_SWE/DailyData_800m'

# Five key WY2023 dates mapped in Lesson 1 and Lesson 2
SEASON_DATES = {
    'Early accumulation': '2022-12-15',
    'Mid-winter build':   '2023-01-31',
    'WY2023 Peak SWE':    '2023-04-01',
    'Active spring melt': '2023-04-12',
    'Late meltout':       '2023-05-01',
}

# Match Lesson 1's Drive setting if running across separate Colab browser tabs
USE_GOOGLE_DRIVE = False
try:
    IN_COLAB = importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/module10_snow_data')
    OUTPUT_DIR = DATA_DIR / 'outputs'
else:
    DATA_DIR = Path('data')
    OUTPUT_DIR = Path('module10_outputs')

# Create output and live-NetCDF cache folders if they do not exist yet
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LIVE_NC_DIR = OUTPUT_DIR / 'ua_live_nc'
LIVE_NC_DIR.mkdir(parents=True, exist_ok=True)


def _normalize_upload_name(raw_name: str) -> str:
    # Remove browser duplicate suffixes such as 'file (1).tif' -> 'file.tif'
    p = Path(raw_name)
    stem = re.sub(r'\s*\(\d+\)$', '', p.stem)
    return f"{stem}{p.suffix}"


def _locate_local_file(filename: str) -> Path | None:
    # Search DATA_DIR, local ./data/, /content/, OUTPUT_DIR, and nearby subfolders
    candidates = [
        DATA_DIR / filename,
        Path('data') / filename,
        Path(filename),
        Path('/content/data') / filename,
        Path('/content') / filename,
        OUTPUT_DIR / filename,
        Path('module10_outputs') / filename,
        DATA_DIR / 'outputs' / filename,
    ]
    for path in candidates:
        if path.exists() and path.is_file() and path.stat().st_size > 0:
            return path
    if DATA_DIR.exists():
        for match in DATA_DIR.rglob(filename):
            if match.is_file() and match.stat().st_size > 0:
                return match
    for pat in (f'*/{filename}', f'*/*/{filename}'):
        for match in Path('.').glob(pat):
            if match.is_file() and match.stat().st_size > 0:
                return match
    return None


def _save_colab_uploads(uploaded: dict[str, bytes]) -> list[str]:
    # Write uploaded individual files or unpack uploaded .zip archives directly into DATA_DIR
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    saved: list[str] = []
    for raw_name, content in uploaded.items():
        clean_name = _normalize_upload_name(raw_name)
        raw_cwd_path = Path(raw_name)
        if clean_name.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(content)) as zf:
                for info in zf.infolist():
                    if info.is_dir():
                        continue
                    member_name = Path(info.filename).name
                    if not member_name or member_name.startswith('.') or '__MACOSX' in info.filename:
                        continue
                    dest = DATA_DIR / _normalize_upload_name(member_name)
                    dest.write_bytes(zf.read(info.filename))
                    saved.append(dest.name)
        else:
            dest = DATA_DIR / clean_name
            dest.write_bytes(content)
            saved.append(dest.name)
        # Remove Colab's duplicate copy in /content/ if saved to a different folder
        if raw_cwd_path.exists() and raw_cwd_path.is_file():
            dest_check = DATA_DIR / clean_name
            if raw_cwd_path.resolve() != dest_check.resolve():
                raw_cwd_path.unlink(missing_ok=True)
    return saved


def ensure_course_files(required_files: list[str]) -> dict[str, Path]:
    # Verify all required Lesson 2 files exist locally, or open one Colab upload dialog
    resolved: dict[str, Path] = {}
    missing: list[str] = []
    for fn in required_files:
        hit = _locate_local_file(fn)
        if hit is not None:
            resolved[fn] = hit
        else:
            missing.append(fn)
    if not missing:
        print(f"All {len(resolved)} Lesson 2 file(s) found locally in {next(iter(resolved.values())).parent}/.")
        return resolved
    if IN_COLAB:
        from google.colab import files as colab_files
        print("Missing Lesson 2 file(s):")
        for fn in missing:
            print(f"  - {fn}")
        print("\nSelect these file(s) from your data/ folder (or upload a .zip of data/):")
        uploaded = colab_files.upload()
        if uploaded:
            _save_colab_uploads(uploaded)
        for fn in list(missing):
            hit = _locate_local_file(fn)
            if hit is not None:
                resolved[fn] = hit
                missing.remove(fn)
    if missing:
        print("\nStill missing:")
        for fn in missing:
            print(f"  - {fn}")
        print(f"Run Lesson 1 with BUILD_FULL_SEASON=True or place them in '{DATA_DIR}' and rerun.")
    else:
        print(f"All {len(resolved)} required Lesson 2 file(s) are ready.")
    return resolved


def find_course_file(filename: str, allow_upload: bool = True) -> Path:
    # Return the resolved local Path for a course file, prompting once in Colab if missing
    hit = _locate_local_file(filename)
    if hit is not None:
        return hit
    if IN_COLAB and allow_upload:
        from google.colab import files as colab_files
        print(f"File '{filename}' not found locally. Upload '{filename}' (or a .zip of data/):")
        uploaded = colab_files.upload()
        if uploaded:
            _save_colab_uploads(uploaded)
            hit = _locate_local_file(filename)
            if hit is not None:
                return hit
    raise FileNotFoundError(
        f"Could not find '{filename}' in '{DATA_DIR}', ./data/, or '{OUTPUT_DIR}'."
    )


def fetch_bytes(url: str, timeout: int = 60, retries: int = 3) -> bytes:
    # Download bytes from a URL with up to 3 attempts and exponential backoff wait times
    last_err = None
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            return resp.content
        except requests.RequestException as exc:
            last_err = exc
            if attempt < retries - 1:
                time.sleep(1.5 * (2 ** attempt))
    raise RuntimeError(f"Request failed for {url}: {last_err}")


LESSON2_DATA_FILES = [
    'minnesota_river_wbd_0702.gpkg',
    'ua_800m_mn_river_wy2023_swe.tif',
    'ua_800m_mn_river_wy2023_depth.tif',
]
ensure_course_files(LESSON2_DATA_FILES)
print(f"Data directory   : {DATA_DIR.resolve()}")
print(f"Outputs directory: {OUTPUT_DIR.resolve()}")

## Part 3: Loading Watershed Boundaries, 196-Day UA Stacks, and Live `0 mm` Check

This cell reads the USGS WBD HUC-4 (`0702`) and 12 HUC-8 subbasin boundaries, loads all 196 days of UA 800 m SWE and snow depth, measures `EPSG:4269` pixel widths with `pyproj.Geod`, and runs an optional `0 mm` spot-check on Peak SWE (`2023-04-01`) against a live or cached NetCDF file using Lesson 1's `float64` full-span coordinate calculation.

In [ ]:
# Read outer HUC-4 watershed polygon and 12 HUC-8 subbasin polygons from the GeoPackage
wbd_path = find_course_file('minnesota_river_wbd_0702.gpkg')
huc4_geo = gpd.read_file(wbd_path, layer='huc04')
huc8_geo = gpd.read_file(wbd_path, layer='huc08').sort_values('huc8').reset_index(drop=True)

# Reproject polygons to equal-area EPSG:5070 (meters) so polygon areas are accurate
huc4_proj = huc4_geo.to_crs(TARGET_CRS)
huc8_proj = huc8_geo.to_crs(TARGET_CRS)
huc8_proj['short_id'] = huc8_proj['huc8'].str[-2:]
huc8_proj['area_km2'] = huc8_proj.geometry.area / 1e6
huc8_proj['area_sqmi'] = huc8_proj['area_km2'] * 0.386102
basin_area_km2 = float(huc4_proj.geometry.area.iloc[0] / 1e6)
assert abs(basin_area_km2 - huc8_proj['area_km2'].sum()) < 0.1, "HUC8 sum mismatch!"

# Build 0.15-degree padded EPSG:4269 bounding box matching Lesson 1's NetCDF crop window
w4, s4, e4, n4 = [float(v) for v in huc4_geo.to_crs('EPSG:4269').total_bounds]
PADDED_BBOX_4269 = (w4 - BBOX_PAD_DEG, s4 - BBOX_PAD_DEG, e4 + BBOX_PAD_DEG, n4 + BBOX_PAD_DEG)

print(f"HUC 4: {huc4_geo['huc4'].iloc[0]} - {huc4_geo['name'].iloc[0]} ({basin_area_km2:,.1f} km² across {len(huc8_proj)} HUC-8s)")
print(huc8_proj[['short_id', 'huc8', 'name', 'states', 'area_km2']].round({'area_km2': 1}).to_string(index=False))

# Load 196-day cropped UA 800 m SWE and Depth stacks
ua_swe_tif = find_course_file('ua_800m_mn_river_wy2023_swe.tif')
ua_depth_tif = find_course_file('ua_800m_mn_river_wy2023_depth.tif')

with rasterio.open(ua_swe_tif) as swe_src, rasterio.open(ua_depth_tif) as dep_src:
    # Read ISO date strings from band descriptions and confirm SWE and Depth have matching dates
    dates_str = list(swe_src.descriptions)
    assert dates_str == list(dep_src.descriptions), "SWE and DEPTH band dates differ!"
    # .read() without a band index loads the full 3D (196, rows, cols) stack into memory
    swe_raw_stack = swe_src.read()
    dep_raw_stack = dep_src.read()
    stack_crs = swe_src.crs
    stack_tr = swe_src.transform
    stack_nodata = int(swe_src.nodata)
    swe_tags = swe_src.tags()

# Extract single 2D arrays for the WY2023 Peak SWE and Active Spring Melt demo dates
pk_date_str = SEASON_DATES['WY2023 Peak SWE']
pk_band_idx = dates_str.index(pk_date_str)
peak_raw = swe_raw_stack[pk_band_idx]
melt_raw = swe_raw_stack[dates_str.index(SEASON_DATES['Active spring melt'])]

print(f"\nLoaded UA stacks: shape={swe_raw_stack.shape} ({dates_str[0]} .. {dates_str[-1]}), CRS={stack_crs.to_string()}, NoData={stack_nodata}")
print(f"Provenance tags : {swe_tags.get('source', '')} | {swe_tags.get('time_convention', '')}")


def open_ua_netcdf_var(nc_path: Path, var_name: str, bbox: tuple | None = None):
    """Open SWE or DEPTH from a UA 800 m NetCDF using exact float64 full-span spacing."""
    meta = {}
    try:
        with xr.open_dataset(nc_path, decode_times=False, mask_and_scale=False) as ds:
            da = ds[var_name]
            # Extract numeric time value and drop the length-1 time axis
            if 'time' in da.dims:
                t_raw = float(ds['time'].values[0])
                t_units = str(ds['time'].attrs.get('units', 'days since 1900-01-01'))
                da = da.isel(time=0)
            else:
                t_raw, t_units = np.nan, 'unknown'
            if set(da.dims) == {'lat', 'lon'} and da.dims != ('lat', 'lon'):
                da = da.transpose('lat', 'lon')

            crs_wkt = str(ds['crs'].attrs.get('spatial_ref', 'EPSG:4269')) if 'crs' in ds else 'EPSG:4269'
            # Compute spacing across the full CONUS span in float64 to avoid float32 rounding drift
            lon = ds['lon'].values.astype(np.float64)
            lat = ds['lat'].values.astype(np.float64)
            lon_asc = bool(lon[-1] > lon[0])
            lat_desc = bool(lat[0] > lat[-1])
            dx = float(abs(lon[-1] - lon[0])) / (len(lon) - 1)
            dy = float(abs(lat[-1] - lat[0])) / (len(lat) - 1)
            # Shift from center coordinates by half a pixel to locate the outer upper-left edge
            left_lon = float(lon[0] if lon_asc else lon[-1]) - dx / 2.0
            top_lat = float(lat[0] if lat_desc else lat[-1]) + dy / 2.0
            full_tr = from_origin(left_lon, top_lat, dx, dy)
            raw_nodata = int(da.attrs.get('_FillValue', UA_NODATA))
            meta = {
                'reader': 'xarray/netCDF4',
                'time_raw_value': t_raw,
                'time_units': t_units,
                'full_shape': tuple(da.shape),
            }
            arr_full = da.values
            if not lat_desc:
                arr_full = arr_full[::-1, :]
            if not lon_asc:
                arr_full = arr_full[:, ::-1]
            # Crop array to the integer pixel window covering bbox
            if bbox is not None:
                rw = window_from_bounds(*bbox, transform=full_tr)
                c0, r0 = max(0, math.floor(rw.col_off)), max(0, math.floor(rw.row_off))
                c1 = min(arr_full.shape[1], math.ceil(rw.col_off + rw.width))
                r1 = min(arr_full.shape[0], math.ceil(rw.row_off + rw.height))
                win = Window(c0, r0, c1 - c0, r1 - r0)
                arr = np.asarray(arr_full[r0:r1, c0:c1], dtype=np.int16).copy()
                tr = rasterio.windows.transform(win, full_tr)
            else:
                arr = np.asarray(arr_full, dtype=np.int16).copy()
                tr = full_tr
            return arr, CRS.from_user_input(crs_wkt), tr, raw_nodata, meta
    except Exception:
        # Backup reader via GDAL/rasterio when netCDF4 is unavailable
        with rasterio.open(f"netcdf:{nc_path}:{var_name}") as src:
            meta = {'reader': 'rasterio/GDAL', 'full_shape': (src.height, src.width)}
            if bbox is not None:
                rw = window_from_bounds(*bbox, transform=src.transform)
                c0, r0 = max(0, math.floor(rw.col_off)), max(0, math.floor(rw.row_off))
                c1 = min(src.width, math.ceil(rw.col_off + rw.width))
                r1 = min(src.height, math.ceil(rw.row_off + rw.height))
                win = Window(c0, r0, c1 - c0, r1 - r0)
                arr = src.read(1, window=win).astype(np.int16)
                tr = src.window_transform(win)
            else:
                arr = src.read(1).astype(np.int16)
                tr = src.transform
            return arr, src.crs, tr, int(src.nodata if src.nodata is not None else UA_NODATA), meta


# Live vs. stack consistency check on Peak SWE (2023-04-01)
pk_nc_name = f"UA_SWE_Depth_800m_v1_{pk_date_str.replace('-', '')}_stable.nc"
pk_nc_path = _locate_local_file(pk_nc_name) or (LIVE_NC_DIR / pk_nc_name)
try:
    if not pk_nc_path.exists() or pk_nc_path.stat().st_size == 0:
        pk_url = f"{UA_BASE_URL}/WY2023/{pk_nc_name}"
        pk_nc_path.write_bytes(fetch_bytes(pk_url, timeout=60, retries=2))
    live_pk_raw, live_crs, live_tr, _, live_meta = open_ua_netcdf_var(pk_nc_path, 'SWE', PADDED_BBOX_4269)
    # Confirm affine transform and pixel values match the 196-day GeoTIFF down to 0 mm
    assert np.allclose(tuple(live_tr), tuple(stack_tr), atol=1e-7), "Transform mismatch between live NetCDF and stack!"
    assert live_pk_raw.shape == peak_raw.shape, f"Shape mismatch: {live_pk_raw.shape} vs {peak_raw.shape}"
    max_diff = int(np.max(np.abs(live_pk_raw.astype(np.int32) - peak_raw.astype(np.int32))))
    assert max_diff == 0, f"Course copy differs from live NetCDF by {max_diff} mm!"
    print(f"Live vs. stack pixel check ({live_meta['reader']}) on {pk_date_str}: PASS (max diff = {max_diff} mm)")
except Exception as err:
    print(f"Live NetCDF spot-check skipped offline ({err}); continuing with verified stack.")

# Measure true EPSG:4269 ground dimensions across Minnesota River latitudes
geod = Geod(ellps='GRS80')  # Reference ellipsoid used by NAD83
dlon, dlat = abs(stack_tr.a), abs(stack_tr.e)
print("\nEPSG:4269 Pixel Geometry Across HUC 0702:")
for label, lat_deg in [('Southern edge (IA/MN border)', s4), ('Basin midpoint', 0.5 * (s4 + n4)), ('Northern edge (ND/SD/MN)', n4)]:
    _, _, ew_m = geod.inv(-95.0, lat_deg, -95.0 + dlon, lat_deg)
    _, _, ns_m = geod.inv(-95.0, lat_deg, -95.0, lat_deg + dlat)
    print(f"  {label:<30s} ({lat_deg:.2f}°N): {ew_m:.1f} m E-W x {ns_m:.1f} m N-S "
          f"(area = {ew_m * ns_m / 1e6:.4f} km² vs 0.6400 km² square 800m)")

## Part 4: Equal-Area Reprojection (`EPSG:5070`, 800 m) and Resampling Comparison

This section builds a snapped `EPSG:5070` (`800 m`, `0.64 km²/cell`) target grid using the buffered-window, reproject, and polygon-mask order of operations, checks how many perimeter cells are lost if you clip tightly before reprojecting, and tests `nearest`, `bilinear`, `cubic`, and `average` on Peak SWE (`2023-04-01`) and Active Spring Melt (`2023-04-12`).

In [ ]:
def build_snapped_grid(bounds: tuple[float, float, float, float], res_m: float):
    """Return (Affine, height, width) aligned to exact multiples of `res_m` in EPSG:5070."""
    minx, miny, maxx, maxy = bounds
    # Snap outer coordinates outward to whole multiples of res_m (800 m)
    x0 = math.floor(minx / res_m) * res_m
    y0 = math.floor(miny / res_m) * res_m
    x1 = math.ceil(maxx / res_m) * res_m
    y1 = math.ceil(maxy / res_m) * res_m
    w = int(round((x1 - x0) / res_m))
    h = int(round((y1 - y0) / res_m))
    # Top-left corner (x0, y1) anchors the Affine grid transform
    return from_origin(x0, y1, res_m, res_m), h, w


# Build the equal-area EPSG:5070 target grid and a boolean mask of cells inside HUC 4
dst_tr, dst_h, dst_w = build_snapped_grid(tuple(huc4_proj.total_bounds), TARGET_RES_M)
# invert=False marks polygon interiors False, so '~' flips them to True inside the basin
h4_mask = ~geometry_mask(huc4_proj.geometry, out_shape=(dst_h, dst_w), transform=dst_tr, invert=False)


def reproject_ua_slice(raw_arr: np.ndarray, src_crs, src_tr: Affine, resampling: Resampling) -> np.ndarray:
    """Convert raw int16 UA array (-999=NoData) to float32, reproject to snapped EPSG:5070, and mask."""
    # Replace integer -999 water/border flags with NaN BEFORE interpolating in float32
    src_f = np.where(raw_arr == UA_NODATA, np.nan, raw_arr.astype(np.float32))
    dst = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
    reproject(
        src_f, dst,
        src_transform=src_tr, src_crs=src_crs,
        dst_transform=dst_tr, dst_crs=TARGET_CRS,
        src_nodata=np.nan, dst_nodata=np.nan,
        resampling=resampling,
    )
    # Clip outside the watershed boundary AFTER reprojection so perimeter cells keep buffered neighbors
    dst[~h4_mask] = np.nan
    return dst


METHODS = {
    'nearest': Resampling.nearest,
    'bilinear': Resampling.bilinear,
    'cubic': Resampling.cubic,
    'average': Resampling.average,
}

# Reproject Peak SWE and Active Melt under all 4 resampling options to inspect differences
resamp_stats = []
melt_grids = {}
peak_grids = {}
for date_label, raw_arr in [('Peak SWE', peak_raw), ('Active Melt', melt_raw)]:
    for m_name, alg in METHODS.items():
        grid = reproject_ua_slice(raw_arr, stack_crs, stack_tr, alg)
        if date_label == 'Peak SWE':
            peak_grids[m_name] = grid
        else:
            melt_grids[m_name] = grid
        valid = grid[~np.isnan(grid)]
        resamp_stats.append({
            'Date': date_label,
            'Method': m_name,
            'Valid cells': int(valid.size),
            'Valid area (km²)': round(valid.size * (TARGET_RES_M ** 2) / 1e6, 1),
            'Min (mm)': round(float(valid.min()), 2),
            'Mean (mm)': round(float(valid.mean()), 2),
            'Max (mm)': round(float(valid.max()), 2),
            'Neg pixels (<0)': int((valid < 0).sum()),
        })

resamp_df = pd.DataFrame(resamp_stats)
h4_mask_km2 = h4_mask.sum() * (TARGET_RES_M ** 2) / 1e6
valid_peak_km2 = float(resamp_df.loc[(resamp_df['Date'] == 'Peak SWE') & (resamp_df['Method'] == 'bilinear'), 'Valid area (km²)'].iloc[0])
masked_diff_km2 = basin_area_km2 - valid_peak_km2
print(f"Target snapped grid: {dst_w} cols x {dst_h} rows @ {TARGET_RES_M:.0f} m ({TARGET_CRS})")
print(f"HUC-4 polygon mask : {int(h4_mask.sum()):,} cells = {h4_mask_km2:,.1f} km² "
      f"({masked_diff_km2:+.1f} km² vs WBD {basin_area_km2:,.1f} km², reflecting UA -999 lake masks)\n")
print(resamp_df.to_string(index=False))

# Compare Mask-Before-Reproject vs. Buffer-Then-Reproject along the watershed edge
tight_4269_mask = ~geometry_mask(
    huc4_geo.to_crs(stack_crs).geometry, out_shape=peak_raw.shape, transform=stack_tr, invert=False
)
# Masking tightly in native EPSG:4269 drops neighbor support along the watershed border during bilinear resampling
tight_src = np.where((peak_raw == UA_NODATA) | (~tight_4269_mask), np.nan, peak_raw.astype(np.float32))
tight_dst = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
reproject(
    tight_src, tight_dst,
    src_transform=stack_tr, src_crs=stack_crs,
    dst_transform=dst_tr, dst_crs=TARGET_CRS,
    src_nodata=np.nan, dst_nodata=np.nan,
    resampling=Resampling.bilinear,
)
tight_dst[~h4_mask] = np.nan
# Count cells that had valid data with our buffer yet became NaN without it
eroded_cells = int(np.sum(~np.isnan(peak_grids['bilinear']) & np.isnan(tight_dst)))
print(f"\nEdge check: Masking tightly in EPSG:4269 BEFORE bilinear reprojection drops "
      f"{eroded_cells:,} perimeter cells ({eroded_cells * 0.64:.1f} km²).")

### Mapping the Resampling Differences (Figure 1)

Figure 1 plots all four resampling outputs on the active spring melt date (`2023-04-12`) next to `Cubic - Bilinear` so you can see where cubic spline ringing produces slight negative undershoots near melting snow edges.

In [ ]:
# Figure 1: Resampling comparison on Active Spring Melt date (Arial)
fig, axes = plt.subplots(1, 5, figsize=(17.5, 4.2), sharey=True)
# Raster image extent [left, right, bottom, top] in EPSG:5070 meters
ext_5070 = [dst_tr.c, dst_tr.c + dst_w * dst_tr.a, dst_tr.f + dst_h * dst_tr.e, dst_tr.f]
# Round peak SWE value up to a clean multiple of 10 mm for the shared color bar
vmax_melt = float(np.ceil(np.nanmax(melt_grids['nearest']) / 10.0) * 10.0)

# Left 4 panels: plot each resampling algorithm on the same color range
for ax, m_name in zip(axes[:4], ['nearest', 'bilinear', 'cubic', 'average']):
    im = ax.imshow(melt_grids[m_name], cmap=SNOW_CMAP, vmin=0, vmax=vmax_melt, extent=ext_5070)
    huc4_proj.boundary.plot(ax=ax, color='#1C2833', lw=1.0)
    v = melt_grids[m_name][~np.isnan(melt_grids[m_name])]
    ax.set_title(f"{m_name.title()}\nmean={v.mean():.1f}, max={v.max():.1f} mm", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(im, ax=axes[:4], shrink=0.82, pad=0.02, label='SWE (mm)')

# 5th panel: pixel-wise difference highlighting cubic overshoot/undershoot against bilinear
diff = melt_grids['cubic'] - melt_grids['bilinear']
im_diff = axes[4].imshow(diff, cmap=DIFF_CMAP, vmin=-2.5, vmax=2.5, extent=ext_5070)
huc4_proj.boundary.plot(ax=axes[4], color='#1C2833', lw=1.0)
axes[4].set_title(f"Cubic - Bilinear\nmin={np.nanmin(diff):.1f}, max={np.nanmax(diff):+.1f} mm", fontsize=10)
axes[4].set_xticks([])
axes[4].set_yticks([])
fig.colorbar(im_diff, ax=axes[4], shrink=0.82, pad=0.05, label='Difference (mm)')

fig.suptitle(
    f"UA 800 m Resampling Comparison During Active Spring Melt ({SEASON_DATES['Active spring melt']})",
    fontsize=12.5, fontweight='bold', x=0.5, y=1.02
)
fig.text(
    0.01, -0.05,
    'Projection: EPSG:5070 (800 m) | Cubic creates spline ringing and slight negative SWE undershoots at the snowline',
    fontsize=8.5, color='#566573'
)
fig1_path = OUTPUT_DIR / 'ua_fig1_resampling_comparison.png'
fig.savefig(fig1_path, dpi=300)
plt.show()
print(f"Saved: {fig1_path}")

Ask your AI assistant: *"Look at the resampling table and Figure 1 above: `cubic` exceeded `nearest` maximum SWE and created small negative values along the snow edge, while `bilinear` stayed inside `[0, raw_max]`. When is `bilinear` preferred over `nearest` or `cubic` for SWE?"*

## Part 5: Full WY2023 Snow Season: HUC-4 and 12 HUC-8 Subbasin Time Series

This cell reprojects all 196 daily bands (`2022-11-01` to `2023-05-15`) of SWE and snow depth onto `EPSG:5070` (`800 m`, `bilinear`), runs five automated PASS/FAIL QA/QC assertions, and ranks the 12 HUC-8 subbasins by WY2023 Peak SWE and snow water volume.

In [ ]:
# Precompute one boolean pixel mask per HUC-8 subbasin on the 800 m EPSG:5070 grid
h8_masks = [
    ~geometry_mask([geom], out_shape=(dst_h, dst_w), transform=dst_tr, invert=False)
    for geom in huc8_proj.geometry
]
cell_area_km2 = (TARGET_RES_M ** 2) / 1e6                # 800 m x 800 m = 0.64 km^2 per cell
basin_area_m2 = float(huc4_proj.geometry.area.iloc[0])

# Preallocate 3D output arrays (196 days, rows, cols) filled with NaN
n_days = len(dates_str)
swe_5070_stack = np.full((n_days, dst_h, dst_w), np.nan, dtype=np.float32)
dep_5070_stack = np.full((n_days, dst_h, dst_w), np.nan, dtype=np.float32)

basin_records = []
subbasin_records = []

# Loop over all 196 daily bands: reproject SWE & Depth and record basin/subbasin stats
for i, d_str in enumerate(dates_str):
    swe_g = reproject_ua_slice(swe_raw_stack[i], stack_crs, stack_tr, Resampling.bilinear)
    dep_g = reproject_ua_slice(dep_raw_stack[i], stack_crs, stack_tr, Resampling.bilinear)
    swe_5070_stack[i] = swe_g
    dep_5070_stack[i] = dep_g

    # Keep all finite land cells inside HUC-4 (including 0 mm snow-free land)
    s_vals = swe_g[h4_mask]
    d_vals = dep_g[h4_mask]
    s_valid = s_vals[~np.isnan(s_vals)]
    d_valid = d_vals[~np.isnan(d_vals)]

    mean_swe = float(s_valid.mean())
    mean_dep = float(d_valid.mean())
    # Convert basin-mean SWE (mm -> m) * area (m^2) -> cubic meters -> km^3 and kAF
    vol_km3 = (mean_swe / 1000.0) * basin_area_m2 / 1e9
    vol_kaf = vol_km3 * 810.7132

    basin_records.append({
        'date': pd.to_datetime(d_str),
        'mean_swe_mm': round(mean_swe, 3),
        'mean_swe_in': round(mean_swe / 25.4, 4),
        'max_swe_mm': round(float(s_valid.max()), 2),
        'mean_depth_mm': round(mean_dep, 3),
        'mean_depth_in': round(mean_dep / 25.4, 4),
        # Bulk density (kg/m^3) is stable only once average snow depth reaches at least 25 mm
        'bulk_density_kg_m3': round(1000.0 * mean_swe / mean_dep, 1) if mean_dep >= 25.0 else np.nan,
        'snow_cover_pct': round(float((s_valid > 1.0).mean() * 100.0), 2),
        'swe_volume_km3': round(vol_km3, 4),
        'swe_volume_kaf': round(vol_kaf, 2),
        'valid_cells': int(s_valid.size),
        'valid_area_km2': round(s_valid.size * cell_area_km2, 2),
    })

    # Compute daily spatial means inside each of the 12 HUC-8 subbasins
    for j, m_sub in enumerate(h8_masks):
        sv = swe_g[m_sub]
        dv = dep_g[m_sub]
        sv = sv[~np.isnan(sv)]
        dv = dv[~np.isnan(dv)]
        sub_swe = float(sv.mean())
        sub_dep = float(dv.mean())
        sub_area_m2 = float(huc8_proj.iloc[j].geometry.area)
        sub_kaf = ((sub_swe / 1000.0) * sub_area_m2 / 1e9) * 810.7132
        subbasin_records.append({
            'date': pd.to_datetime(d_str),
            'short_id': huc8_proj.iloc[j]['short_id'],
            'huc8': huc8_proj.iloc[j]['huc8'],
            'name': huc8_proj.iloc[j]['name'],
            'mean_swe_mm': round(sub_swe, 3),
            'mean_swe_in': round(sub_swe / 25.4, 4),
            'mean_depth_mm': round(sub_dep, 3),
            'snow_cover_pct': round(float((sv > 1.0).mean() * 100.0), 2),
            'swe_volume_kaf': round(sub_kaf, 2),
        })

basin_df = pd.DataFrame(basin_records)
sub_df = pd.DataFrame(subbasin_records)

# Automated PASS/FAIL QA/QC checks
valid_area_pct = float(basin_df['valid_area_km2'].iloc[0] / basin_area_km2 * 100.0)
qa_checks = [
    ('Target CRS is EPSG:5070', CRS.from_user_input(TARGET_CRS).to_epsg() == 5070, TARGET_CRS),
    ('Day count complete (196 days)', len(basin_df) == 196, f"{len(basin_df)} days"),
    ('Zero negative SWE pixels', bool(np.nanmin(swe_5070_stack) >= 0.0), f"min={np.nanmin(swe_5070_stack):.3f}"),
    ('Zero negative Depth pixels', bool(np.nanmin(dep_5070_stack) >= 0.0), f"min={np.nanmin(dep_5070_stack):.3f}"),
    ('Valid area >= 99% (minus lakes)', valid_area_pct >= 99.0, f"{valid_area_pct:.2f}%"),
]
print("QA/QC Verification:")
for label, ok, detail in qa_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label:<36s} -> {detail}")
    assert ok, f"QA check failed: {label}"

# Locate Peak SWE row across the 196-day basin series
pk_row = basin_df.loc[basin_df['mean_swe_mm'].idxmax()]
print(f"\nWY2023 Basin Peak SWE: {pk_row['mean_swe_mm']:.1f} mm ({pk_row['mean_swe_in']:.2f} in) on "
      f"{pk_row['date'].strftime('%Y-%m-%d')} | Volume = {pk_row['swe_volume_km3']:.3f} km³ ({pk_row['swe_volume_kaf']:,.0f} kAF)")

# Summarize Peak SWE, volume, and spring meltout date (< 5 mm) per HUC-8 subbasin
sub_summary_rows = []
for huc8_id, grp in sub_df.groupby('huc8', sort=True):
    grp = grp.sort_values('date').reset_index(drop=True)
    pk_i = grp['mean_swe_mm'].idxmax()
    pk = grp.loc[pk_i]
    post_pk = grp.loc[pk_i:]  # Slice from peak date onward to detect meltout
    melt_done = post_pk[post_pk['mean_swe_mm'] < 5.0]
    meltout = melt_done.iloc[0]['date'].strftime('%Y-%m-%d') if not melt_done.empty else 'After window'
    sub_summary_rows.append({
        'ID': pk['short_id'],
        'HUC-8': huc8_id,
        'Subbasin': pk['name'],
        'Peak SWE (mm)': round(pk['mean_swe_mm'], 1),
        'Peak SWE (in)': round(pk['mean_swe_in'], 2),
        'Peak Date': pk['date'].strftime('%Y-%m-%d'),
        'Peak Depth (mm)': round(pk['mean_depth_mm'], 1),
        'Peak Vol (kAF)': round(pk['swe_volume_kaf'], 1),
        'Meltout (<5 mm)': meltout,
    })

sub_peak_df = pd.DataFrame(sub_summary_rows).sort_values('Peak SWE (mm)', ascending=False).reset_index(drop=True)
print("\n12 HUC-8 Subbasins Ranked by Peak SWE (WY2023):")
print(sub_peak_df.to_string(index=False))

## Part 6: Snowpack Maps and Seasonal Hydrographs

The next three cells generate 300 dpi Arial maps and charts:

- Figure 2 maps Peak SWE and Snow Depth (`2023-04-01`) across HUC 4 (`0702`) with HUC-8 boundaries.
- Figure 3 places four dates from early winter through spring melt on one shared color bar (`0 to 225 mm`).
- Figure 4 plots the daily HUC-4 SWE and Depth time series, 12 HUC-8 curves, and ranked Peak Snow Water Volume (`kAF`).

In [ ]:
# Figure 2: Dual-panel Peak SWE and Peak Snow Depth Map
x_min, y_min, x_max, y_max = huc4_proj.total_bounds
pk_swe_grid = swe_5070_stack[pk_band_idx]
pk_dep_grid = dep_5070_stack[pk_band_idx]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.0), sharey=True)
# Use the 99.5th percentile rounded up so isolated high cells do not flatten contrast across the basin
swe_vmax = float(np.ceil(np.nanpercentile(pk_swe_grid, 99.5) / 25.0) * 25.0)
dep_vmax = float(np.ceil(np.nanpercentile(pk_dep_grid, 99.5) / 50.0) * 50.0)

panels = [
    (axes[0], pk_swe_grid, 'Snow Water Equivalent (mm)', swe_vmax,
     f"Peak SWE ({pk_row['date'].strftime('%Y-%m-%d')})\nBasin Mean: {pk_row['mean_swe_mm']:.1f} mm ({pk_row['mean_swe_in']:.2f} in) | {pk_row['swe_volume_kaf']:,.0f} kAF"),
    (axes[1], pk_dep_grid, 'Snow Depth (mm)', dep_vmax,
     f"Snow Depth on Peak SWE Date ({pk_row['date'].strftime('%Y-%m-%d')})\nBasin Mean: {pk_row['mean_depth_mm']:.0f} mm ({pk_row['mean_depth_in']:.1f} in)"),
]

for idx, (ax, arr, clabel, vmax, title) in enumerate(panels):
    im = ax.imshow(arr, cmap=SNOW_CMAP, vmin=0, vmax=vmax, extent=ext_5070)
    huc8_proj.boundary.plot(ax=ax, color='#566573', lw=0.75, alpha=0.85)
    huc4_proj.boundary.plot(ax=ax, color='#0B2545', lw=2.0)
    # Place short HUC-8 numeric badges ('01' to '12') inside each subbasin
    for _, r in huc8_proj.iterrows():
        pt = r.geometry.representative_point()
        lx, ly = pt.x, pt.y + (18_000 if r['short_id'] == '02' else 0)
        ax.text(lx, ly, r['short_id'], ha='center', va='center', fontsize=8,
                fontweight='bold', color='#1C2833',
                bbox=dict(boxstyle='round,pad=0.14', facecolor='white', edgecolor='none', alpha=0.78))
    ax.set_xlim(x_min - 18_000, x_max + 18_000)
    ax.set_ylim(y_min - 22_000, y_max + 18_000)
    ax.set_aspect('equal')
    add_scalebar(ax, length_km=50, location=(0.05, 0.05))
    add_north_arrow(ax, x_frac=0.93, y_frac=0.87)
    format_km_axes(ax)
    if idx == 1:
        ax.set_ylabel('')  # Omit repeated y-axis label on the right map
    ax.set_title(title, fontsize=10.5, pad=8)
    fig.colorbar(im, ax=ax, shrink=0.84, pad=0.02, label=clabel)

fig.suptitle('Minnesota River Basin (HUC 0702): WY2023 Peak Snowpack Conditions (UA 800 m)',
             fontsize=13, fontweight='bold', x=0.5, y=0.98)
fig.subplots_adjust(left=0.06, right=0.97, top=0.86, bottom=0.12, wspace=0.10)
fig.text(0.01, 0.01,
         'Data: University of Arizona DailyData_800m (stable) | Reprojected: bilinear to EPSG:5070 (800 m) | Water masks (-999) excluded',
         fontsize=8.5, color='#566573')
fig2_path = OUTPUT_DIR / 'ua_fig2_peak_swe_depth_map.png'
fig.savefig(fig2_path, dpi=300)
plt.show()
print(f"Saved: {fig2_path}")

In [ ]:
# Figure 3: 4-Panel Seasonal Snow Evolution on a Shared Color Scale
evolution_items = [
    ('Early Winter', SEASON_DATES['Early accumulation']),
    ('Mid-Winter', SEASON_DATES['Mid-winter build']),
    ('Peak SWE', SEASON_DATES['WY2023 Peak SWE']),
    ('Spring Melt', SEASON_DATES['Active spring melt']),
]

fig, axes = plt.subplots(1, 4, figsize=(16.5, 4.4), sharey=True)
# Plot 4 seasonal dates using the exact same color limits (0 to swe_vmax)
for idx, (ax, (stage, d_str)) in enumerate(zip(axes, evolution_items)):
    b_idx = dates_str.index(d_str)
    arr = swe_5070_stack[b_idx]
    row = basin_df.loc[basin_df['date'] == pd.Timestamp(d_str)].iloc[0]
    im = ax.imshow(arr, cmap=SNOW_CMAP, vmin=0, vmax=swe_vmax, extent=ext_5070)
    huc8_proj.boundary.plot(ax=ax, color='#566573', lw=0.6, alpha=0.8)
    huc4_proj.boundary.plot(ax=ax, color='#0B2545', lw=1.6)
    ax.set_xlim(x_min - 15_000, x_max + 15_000)
    ax.set_ylim(y_min - 20_000, y_max + 15_000)
    ax.set_aspect('equal')
    # Place scale bar and north arrow only on the first panel to keep all 4 panels clean
    if idx == 0:
        add_scalebar(ax, length_km=50, location=(0.06, 0.05), show_mid=False)
        add_north_arrow(ax, x_frac=0.91, y_frac=0.86)
    format_km_axes(ax)
    if idx > 0:
        ax.set_ylabel('')
    ax.set_title(f"{stage}: {d_str}\nMean {row['mean_swe_mm']:.1f} mm | SCA {row['snow_cover_pct']:.0f}%", fontsize=10)

# Attach one shared color bar across all 4 panels
fig.colorbar(im, ax=axes, shrink=0.82, pad=0.015, label='Snow Water Equivalent (mm)')
fig.suptitle('WY2023 Spatial SWE Progression Across Minnesota River HUC 4 (0702)',
             fontsize=12.5, fontweight='bold', x=0.5, y=1.04)
fig.text(0.01, -0.05,
         f'All panels share a single 0-{swe_vmax:.0f} mm color scale so accumulation and ablation patterns compare directly.',
         fontsize=8.5, color='#566573')
fig3_path = OUTPUT_DIR / 'ua_fig3_seasonal_evolution.png'
fig.savefig(fig3_path, dpi=300)
plt.show()
print(f"Saved: {fig3_path}")

In [ ]:
# Figure 4: Basin-Mean SWE/Depth Hydrograph + Subbasin Trajectories + Ranked Peak Volume
# GridSpec creates a wide top time-series panel (gs[0, :]) over two side-by-side bottom panels
fig = plt.figure(figsize=(13.5, 8.2))
gs = fig.add_gridspec(2, 2, height_ratios=[1.05, 1.0], hspace=0.35, wspace=0.25)
ax_ts = fig.add_subplot(gs[0, :])
ax_sub = fig.add_subplot(gs[1, 0])
ax_bar = fig.add_subplot(gs[1, 1])

# Top panel: plot SWE on left y-axis and Snow Depth on right twin y-axis
l1, = ax_ts.plot(basin_df['date'], basin_df['mean_swe_mm'], color='#1B4F72', lw=2.3, label='Basin-Mean SWE (mm)')
ax_ts.fill_between(basin_df['date'], 0, basin_df['mean_swe_mm'], color='#5DADE2', alpha=0.25)
ax_ts.set_ylabel('Snow Water Equivalent (mm)', color='#1B4F72', fontweight='bold')
ax_ts.tick_params(axis='y', labelcolor='#1B4F72')
ax_ts.set_ylim(0, basin_df['mean_swe_mm'].max() * 1.18)

ax_dep = ax_ts.twinx()  # Shared x-axis, separate right-hand y-axis for Snow Depth
ax_dep.spines['top'].set_visible(False)
l2, = ax_dep.plot(basin_df['date'], basin_df['mean_depth_mm'], color='#7D3C98', lw=1.7, ls='--', label='Basin-Mean Snow Depth (mm)')
ax_dep.set_ylabel('Snow Depth (mm)', color='#7D3C98', fontweight='bold')
ax_dep.tick_params(axis='y', labelcolor='#7D3C98')
ax_dep.set_ylim(0, basin_df['mean_depth_mm'].max() * 1.18)

# Mark the Peak SWE date with a vertical dotted line and callout box
ax_ts.axvline(pk_row['date'], color='#1B4F72', ls=':', lw=1.2)
ax_ts.annotate(
    f"Peak SWE: {pk_row['mean_swe_mm']:.1f} mm ({pk_row['mean_swe_in']:.2f} in)\n"
    f"{pk_row['date'].strftime('%Y-%m-%d')} | {pk_row['swe_volume_kaf']:,.0f} kAF",
    xy=(pk_row['date'], pk_row['mean_swe_mm']),
    xytext=(-130, 14), textcoords='offset points',
    fontsize=9, fontweight='bold', color='#1B4F72',
    arrowprops=dict(arrowstyle='->', color='#1B4F72', lw=1.1),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#AED6F1')
)
ax_ts.xaxis.set_major_locator(mdates.MonthLocator())
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_ts.grid(alpha=0.25)
ax_ts.legend([l1, l2], [l1.get_label(), l2.get_label()], loc='upper left', frameon=True)
ax_ts.set_title('Minnesota River HUC 4 (0702): Daily Basin-Mean SWE and Snow Depth (WY2023)')

# Bottom-left panel: highlight highest and lowest HUC-8 curves against faint background curves
top_id = sub_peak_df.iloc[0]['HUC-8']
bot_id = sub_peak_df.iloc[-1]['HUC-8']
for huc8_id, grp in sub_df.groupby('huc8'):
    grp = grp.sort_values('date')
    name = grp['name'].iloc[0]
    if huc8_id == top_id:
        ax_sub.plot(grp['date'], grp['mean_swe_mm'], color='#0E6251', lw=2.2, label=f"Highest: {name}")
    elif huc8_id == bot_id:
        ax_sub.plot(grp['date'], grp['mean_swe_mm'], color='#B03A2E', lw=2.0, label=f"Lowest: {name}")
    else:
        ax_sub.plot(grp['date'], grp['mean_swe_mm'], color='#85929E', lw=0.9, alpha=0.55)
ax_sub.plot(basin_df['date'], basin_df['mean_swe_mm'], color='#1B4F72', lw=2.0, ls='--', label='HUC-4 Basin Mean')
ax_sub.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax_sub.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax_sub.set_ylabel('Mean SWE (mm)')
ax_sub.set_xlabel('WY2023 Snow Season')
ax_sub.set_title('HUC-8 Subbasin SWE Spread')
ax_sub.grid(alpha=0.25)
ax_sub.legend(loc='upper left', fontsize=8.5, frameon=True)

# Bottom-right panel: horizontal bar chart ranked by peak snow water storage (kAF)
bar_df = sub_peak_df.sort_values('Peak Vol (kAF)', ascending=True)
colors_b = plt.cm.Blues(np.linspace(0.35, 0.9, len(bar_df)))
bars = ax_bar.barh(bar_df['Subbasin'], bar_df['Peak Vol (kAF)'], color=colors_b, edgecolor='#1B4F72', lw=0.6)
for bar, (_, r) in zip(bars, bar_df.iterrows()):
    ax_bar.text(bar.get_width() + 8, bar.get_y() + bar.get_height() / 2,
                f"{r['Peak Vol (kAF)']:.0f} kAF ({r['Peak SWE (mm)']:.0f} mm)",
                va='center', fontsize=8)
ax_bar.set_xlim(0, bar_df['Peak Vol (kAF)'].max() * 1.38)
ax_bar.set_xlabel('Peak Snow Water Volume (thousand acre-feet, kAF)')
ax_bar.set_title('Peak Snow Water Storage by Subbasin')
ax_bar.grid(axis='x', alpha=0.25)

fig.text(
    0.01, 0.01,
    'Data: University of Arizona DailyData_800m (stable) | Reprojected: bilinear to EPSG:5070 (800 m)',
    fontsize=8.5, color='#566573'
)
fig.subplots_adjust(bottom=0.09)
fig4_path = OUTPUT_DIR / 'ua_fig4_wy2023_timeseries_subbasins.png'
fig.savefig(fig4_path, dpi=300)
plt.show()
print(f"Saved: {fig4_path}")

## Part 7: Exporting Traceable Tables and Peak GeoTIFF

The cell below writes daily HUC-4 and HUC-8 CSV summaries, the subbasin peak table, and the `EPSG:5070` (`800 m`) Peak SWE GeoTIFF into `OUTPUT_DIR`. Lesson 3 reads `ua_wy2023_huc04_daily_timeseries.csv` and `ua_wy2023_huc08_daily_timeseries.csv` directly when comparing SNODAS against UA 800 m.

In [ ]:
# Record dataset, CRS, resampling, and access date metadata inside the exported CSVs and GeoTIFF
provenance_note = (
    f"UA DailyData_800m (stable); DOI:10.5067/0GGPB220EX6A; "
    f"Target:{TARGET_CRS} {TARGET_RES_M:.0f}m bilinear; Accessed:{datetime.date.today().isoformat()}"
)

basin_out = OUTPUT_DIR / 'ua_wy2023_huc04_daily_timeseries.csv'
sub_out = OUTPUT_DIR / 'ua_wy2023_huc08_daily_timeseries.csv'
peak_tbl_out = OUTPUT_DIR / 'ua_wy2023_huc08_peak_summary.csv'
peak_tif_out = OUTPUT_DIR / 'ua_wy2023_peak_swe_5070_800m.tif'

# Copy DataFrames before adding provenance so in-memory analysis tables stay clean
basin_export = basin_df.copy()
basin_export['source_note'] = provenance_note
basin_export.to_csv(basin_out, index=False)

sub_export = sub_df.copy()
sub_export['source_note'] = provenance_note
sub_export.to_csv(sub_out, index=False)

sub_peak_df.to_csv(peak_tbl_out, index=False)

# Replace NaN with -9999.0 before writing the clipped Peak SWE float32 GeoTIFF
tif_arr = np.where(np.isnan(pk_swe_grid), -9999.0, pk_swe_grid).astype(np.float32)
with rasterio.open(
    peak_tif_out, 'w',
    driver='GTiff', height=dst_h, width=dst_w, count=1, dtype='float32',
    crs=TARGET_CRS, transform=dst_tr, nodata=-9999.0,
    compress='deflate', predictor=2, tiled=True,
) as dst:
    dst.write(tif_arr, 1)
    dst.set_band_description(1, f"UA_800m_SWE_{SEASON_DATES['WY2023 Peak SWE']}")
    dst.update_tags(
        source='University of Arizona DailyData_800m',
        date=SEASON_DATES['WY2023 Peak SWE'],
        units='millimeters',
        resampling='bilinear',
        provenance=provenance_note,
    )

for p in (basin_out, sub_out, peak_tbl_out, peak_tif_out):
    print(f"Exported: {p} ({p.stat().st_size / 1024:.1f} KB)")

### Optional: Inspect Saved File Locations and Download Lesson 2 Outputs

All Lesson 2 tables, the Peak GeoTIFF, and figures are stored inside `OUTPUT_DIR`:

- In Google Colab, click the **Files** folder icon on the left sidebar and open `/content/module10_outputs/` (or `/content/drive/MyDrive/module10_snow_data/outputs/` when `USE_GOOGLE_DRIVE = True`) to browse and right-click individual files.
- Run the cell below to list every saved output path and file size, and bundle them into `lesson2_ua_outputs.zip`.
- Set `DOWNLOAD_OUTPUTS_ZIP = True` before running if you want Colab to download `lesson2_ua_outputs.zip` directly to your computer.

In [ ]:
DOWNLOAD_OUTPUTS_ZIP = False  # Set True in Google Colab to download lesson2_ua_outputs.zip

# Gather all Lesson 2 CSV, GeoTIFF, and figure paths
output_files = [
    basin_out, sub_out, peak_tbl_out, peak_tif_out,
    fig1_path, fig2_path, fig3_path, fig4_path,
]

print("Lesson 2 saved files:")
for out_f in output_files:
    assert out_f.exists(), f"Missing output: {out_f}"
    print(f"  {out_f.resolve()} ({out_f.stat().st_size / 1024:,.1f} KB)")

# Compress all outputs into a single zip archive inside OUTPUT_DIR
zip_path = OUTPUT_DIR / 'lesson2_ua_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for out_f in output_files:
        zf.write(out_f, arcname=out_f.name)
print(f"\nPackaged zip archive: {zip_path.resolve()} ({zip_path.stat().st_size / 1024:,.1f} KB)")

# Trigger a browser download when running inside Google Colab
if DOWNLOAD_OUTPUTS_ZIP:
    if IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
    else:
        print("Running locally: copy or open the .zip path printed above.")

## Part 8: Practice Debugging 1: Missing `-999` Mask and Zero-Snow Filter

The script below runs without raising any Python errors while computing spring-melt SWE over the Minnesota River basin. Two bugs work together to distort both the map and the basin-mean SWE. This cell runs on in-memory arrays and leaves your saved outputs untouched.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING 1: THIS CELL CONTAINS TWO INTENTIONAL BUGS.
# It runs without an error. Diagnose what went wrong and why.
# ---------------------------------------------------------------
# Bug A: Casting raw int16 directly to float WITHOUT masking -999 first
buggy_src = melt_raw.astype(np.float32)
buggy_dst = np.zeros((dst_h, dst_w), dtype=np.float32)

reproject(
    buggy_src, buggy_dst,
    src_transform=stack_tr, src_crs=stack_crs,
    dst_transform=dst_tr, dst_crs=TARGET_CRS,
    resampling=Resampling.bilinear,   # Oops: src_nodata is not set
)
buggy_basin = buggy_dst[h4_mask]

# Bug B: Dropping zero-SWE pixels so bare ground is excluded from the basin mean
snow_only_mean = float(buggy_basin[buggy_basin > 0].mean())
true_mean = float(basin_df.loc[basin_df['date'] == pd.Timestamp(SEASON_DATES['Active spring melt']), 'mean_swe_mm'].iloc[0])

print(f"Melt date: {SEASON_DATES['Active spring melt']}")
print(f"Buggy grid min SWE : {buggy_basin.min():.2f} mm  (negative pixels: {(buggy_basin < 0).sum():,})")
print(f"Buggy 'basin mean' : {snow_only_mean:.2f} mm")
print(f"True basin mean    : {true_mean:.2f} mm  (error: {snow_only_mean - true_mean:+.2f} mm)")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
im = ax.imshow(np.where(h4_mask, buggy_dst, np.nan), cmap='RdBu', vmin=-250, vmax=250, extent=ext_5070)
huc4_proj.boundary.plot(ax=ax, color='black', lw=1.2)
ax.set_title("Buggy Reprojection: -999 Water-Mask Bleed (Red = Negative SWE)")
format_km_axes(ax)
fig.colorbar(im, ax=ax, shrink=0.82, label='SWE (mm)')
plt.show()

### Your Task (1)

1. Identify where the red negative-SWE pockets (down to `-999 mm`) inside the watershed originate using what `-999` denotes in `SWE_Mask_800m.nc`.
2. Explain why `buggy_basin[buggy_basin > 0].mean()` overstates the true basin mean and distinguish `mean(SWE > 0)` from `mean(all valid pixels)`.

Suggested AI prompt:

*"My reprojected UA 800 m SWE raster has negative values down to `-999 mm` around Minnesota lakes, and my spring-melt basin mean is higher than expected when I filter `swe > 0`. Trace both mistakes and fix them."*

Write your fix below and verify zero negative pixels and agreement with `true_mean`.

In [ ]:
# YOUR FIX: mask UA_NODATA (-999) before bilinear reprojection and average all valid pixels (including 0 mm).
#
# Hints:
# 1. Replace UA_NODATA (-999) with np.nan BEFORE calling reproject(..., src_nodata=np.nan, dst_nodata=np.nan).
# 2. Average over all finite (~np.isnan) pixels inside h4_mask so 0 mm bare-ground cells remain in the denominator.

## Part 9: Practice Debugging 2: Geographic Volume Bias and Unchecked Cubic Overshoot

This cell demonstrates two additional errors common in snow-volume scripts:

- Bug A multiplies valid `EPSG:4269` pixel counts by `0.64 km²` (`800 m x 800 m`) directly without reprojecting to `EPSG:5070`. Because native Minnesota pixels average `~659 m x 926 m` (`~0.610 km²`), counting `EPSG:4269` cells as `0.64 km²` squares inflates watershed area and water volume by ~5% (`~195 kAF`).
- Bug B reprojects active spring melt (`2023-04-12`) using `Resampling.cubic` without non-negativity checks, producing negative SWE values along snow margins.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING 2: THIS CELL CONTAINS TWO INTENTIONAL BUGS.
# ---------------------------------------------------------------
# Bug A: Treating raw EPSG:4269 pixels as 800m x 800m (0.64 km^2) squares
mask_4269 = ~geometry_mask(huc4_geo.to_crs(stack_crs).geometry, out_shape=peak_raw.shape, transform=stack_tr, invert=False)
raw_valid = peak_raw[mask_4269 & (peak_raw != UA_NODATA)]
buggy_area_km2 = raw_valid.size * 0.64
buggy_vol_kaf = ((float(raw_valid.mean()) / 1000.0) * (buggy_area_km2 * 1e6) / 1e9) * 810.7132
true_vol_kaf = float(pk_row['swe_volume_kaf'])

print("Bug A (assuming square 800m cells in EPSG:4269):")
print(f"  Buggy 'watershed area' : {buggy_area_km2:,.1f} km² (vs WBD {basin_area_km2:,.1f} km²)")
print(f"  Buggy Peak Snow Volume : {buggy_vol_kaf:,.1f} kAF (vs {true_vol_kaf:,.1f} kAF -> error {buggy_vol_kaf - true_vol_kaf:+.1f} kAF)")

# Bug B: Unchecked cubic interpolation on Active Melt (creates negative SWE)
cubic_melt = reproject_ua_slice(melt_raw, stack_crs, stack_tr, Resampling.cubic)
neg_count = int((cubic_melt[~np.isnan(cubic_melt)] < 0).sum())
print("\nBug B (Resampling.cubic without QA bounds check):")
print(f"  Min SWE = {np.nanmin(cubic_melt):.3f} mm across {neg_count:,} negative-SWE cells")

### Your Task (2)

1. Explain why multiplying native `EPSG:4269` valid pixel counts by `0.64 km²` raised watershed area above `44,050.3 km²`.
2. Explain why `Resampling.cubic` generates negative values next to `0 mm` bare ground while `Resampling.bilinear` stays inside `[0, raw_max]`.

Suggested AI prompt:

*"My snow volume script computed `valid_pixels * 0.64 km²` on `EPSG:4269` UA grids and inflated Minnesota River area by ~5%, and `Resampling.cubic` gave negative spring-melt SWE. Write a corrected `EPSG:5070` `bilinear` workflow with strict area and non-negativity assertions."*

In [ ]:
# YOUR FIX:
# 1. Compute volume exclusively on snapped EPSG:5070 (800 m = 0.64 km^2) grids via reproject_ua_slice(..., Resampling.bilinear).
# 2. Assert np.nanmin(fixed_grid) >= 0.0 and valid_area_km2 within 1% of basin_area_km2.

## Engineering Interpretation and Limitations

### Strengths of UA 800 m

- Over 40 years of continuous daily grids (WY1982 to present) support multi-decade flood comparisons across WY1997, WY2001, WY2011, WY2019, and WY2023 that predate SNODAS (2003).
- Reprojected onto `EPSG:5070`, HUC-8 volume rankings (`kAF`, `km³`) show which tributaries hold the largest spring snowpack storage.
- Paired `SWE` and `DEPTH` grids allow calculation of bulk snowpack density (`SWE / Depth`) as the pack ripens toward `300 to 400 kg/m³` prior to meltwater release.

### Engineering Cautions

1. At 800 m resolution, individual cells average open wind-scoured fields and drifted ditches together.
2. Minnesota has zero SNOTEL stations, so UA grids depend on NWS COOP snow depth spacing, Dawson density, and PRISM precipitation. Cross-check single-cell extremes against NWS or MNDNR field snow surveys when sizing hydraulic structures.
3. Aligning daily UA SWE (`12:00 UTC` PRISM day) with USGS daily streamflow or SNODAS (`06:00 UTC` in Lesson 3) involves a 6-to-18 hour timestamp offset.
4. Keep lake `-999` masks out of interpolation stencils while retaining `0 mm` bare-ground cells in basin averages.

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---|---|---|
| Negative SWE pixels (`-50` to `-999`) after reprojection | Forgot `src_nodata=np.nan` after masking `-999` | Replace `UA_NODATA (-999)` with `np.nan` before `reproject()`, and use `bilinear` |
| Basin mean spikes high during April melt | Zero-snow pixels excluded (`swe > 0`) | Keep `0 mm` bare-ground cells in the spatial mean denominator |
| Perimeter cells lost around basin edge | Tightly clipped before reprojecting | Retain Lesson 1's `0.15°` geographic margin until after `reproject()` |
| Basin area or volume ~5% too large | Computed `count * 0.64 km²` in `EPSG:4269` | Reproject to snapped equal-area `EPSG:5070` (`800 m`) first |
| `FileNotFoundError` on `ua_800m_*.tif` or `*.gpkg` | Files not yet in `/content/data/` or Google Drive | Run Lesson 1 (`BUILD_FULL_SEASON=True`), upload `data/`, or set `USE_GOOGLE_DRIVE = True` |

## Practice Exercises

### Exercise 1: Track Bulk Snowpack Density Through Spring Ripening

Plot daily basin-mean **bulk snow density** (`bulk_density_kg_m3` in `basin_df`, masked where `mean_depth_mm < 25 mm`) from November through April in Arial font and mark the Peak SWE date (`2023-04-01`). Check whether snowpack density rises toward `300 to 400 kg/m³` prior to rapid melt.

In [ ]:
# EXERCISE 1: Bulk snowpack density (kg/m^3) across WY2023
# Your code here.

### Exercise 2: Compare Two Contrasting HUC-8 Subbasins

Select the highest-peak HUC-8 subbasin (`07020012 Lower Minnesota`) and lowest-peak HUC-8 subbasin (`07020002 Pomme De Terre`) from `sub_df`. Compare their peak SWE timing, peak water volume (`kAF`), days from Peak SWE to `< 5 mm` meltout, and average spring melt rate (`mm/day`).

In [ ]:
# EXERCISE 2: Compare highest vs. lowest peak-SWE HUC-8 subbasins
# Your code here.

### Exercise 3: Map Spatial Bulk Snowpack Density on Peak SWE Date

Using `pk_swe_grid` and `pk_dep_grid` in `EPSG:5070`, compute spatial bulk snow density (`1000 * pk_swe_grid / pk_dep_grid`) masked where `pk_dep_grid >= 50 mm`, and create an Arial map with HUC-8 boundaries, scale bar, and north arrow.

In [ ]:
# EXERCISE 3: Spatial bulk snowpack density (kg/m^3) map on Peak SWE date
# Your code here.

## Lesson 2 Summary

You reprojected Lesson 1's native `EPSG:4269` UA 800 m stacks onto a snapped equal-area `EPSG:5070` (`800 m`) grid, verified `0 mm` pixel agreement against the live NetCDF, evaluated resampling behavior along an active spring snowline, and quantified WY2023 snow water storage across Minnesota River HUC 4 (`123.7 mm`, `4.87 in`, `5.450 km³`, `4,418 kAF` on `2023-04-01`) and all 12 HUC-8 subbasins.

### Next Notebook

Open `10_03_snodas_snow_analysis_mapping.ipynb` (Lesson 3) to analyze NOAA/NOHRSC SNODAS 1 km SWE (`1034`) and 24-hour Snow Melt Runoff (`1044`), reconcile the October 1, 2013 spatial shift, map seasonal snowmelt, and cross-compare `SNODAS - UA` against your Lesson 2 CSV exports.